In [22]:
# ==========================================================
# 5G PATH LOSS PREDICTION USING RANDOM FOREST
# Google Colab Version
# ==========================================================

# Install libraries (Only needed in Google Colab)
!pip install -q joblib

# ==========================================================
# Import Libraries
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

# ==========================================================
# Load Dataset
# ==========================================================

from google.colab import files

uploaded = files.upload()

# Replace filename if needed
df = pd.read_csv("5g-South Asia.csv")

print("Dataset Loaded Successfully!\n")

print(df.head())

# ==========================================================
# Dataset Information
# ==========================================================

print("\nDataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

# ==========================================================
# Remove Extra Spaces from Column Names
# ==========================================================

df.columns = df.columns.str.strip()

# ==========================================================
# Define Target Variable
# ==========================================================

target = "Path Loss (dB)"

X = df.drop(columns=[target])
y = df[target]

# ==========================================================
# Identify Numerical & Categorical Columns
# ==========================================================

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("\nCategorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

# ==========================================================
# Preprocessing Pipeline
# ==========================================================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# ==========================================================
# Train-Test Split (80:20)
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])

# ==========================================================
# Create Model Pipeline
# ==========================================================

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# ==========================================================
# Train Model
# ==========================================================

print("\nTraining Model...")

model.fit(X_train, y_train)

print("Model Training Completed!")

# ==========================================================
# Prediction
# ==========================================================

y_pred = model.predict(X_test)

# ==========================================================
# Model Evaluation
# ==========================================================

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n==============================")
print("Model Evaluation")
print("==============================")

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R² Score : {r2:.4f}")

# ==========================================================
# Save Model
# ==========================================================

joblib.dump(model, "5G_PathLoss_Model.pkl")

print("\nModel Saved Successfully!")
print("Filename : 5G_PathLoss_Model.pkl")

# ==========================================================
# DATA VISUALIZATION
# ==========================================================

sns.set(style="whitegrid")

# Create a directory to save plots
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

# ----------------------------------------------------------
# 1. Target Distribution
# ----------------------------------------------------------

plt.figure(figsize=(8,5))
sns.histplot(df[target], bins=30, kde=True)
plt.title("Distribution of Path Loss")
plt.savefig(os.path.join(results_dir, "target_distribution.png"))
plt.close()

# ----------------------------------------------------------
# 2. Correlation Heatmap
# ----------------------------------------------------------

plt.figure(figsize=(12,8))
sns.heatmap(df.select_dtypes(include=np.number).corr(),
            annot=True,
            cmap="coolwarm",
            fmt=".2f")
plt.title("Correlation Heatmap")
plt.savefig(os.path.join(results_dir, "correlation_heatmap.png"))
plt.close()

# ----------------------------------------------------------
# 3. Path Loss vs Distance
# ----------------------------------------------------------

plt.figure(figsize=(8,5))
sns.scatterplot(
    x=df["T-R Separation Distance (m)"],
    y=df[target]
)
plt.title("Distance vs Path Loss")
plt.savefig(os.path.join(results_dir, "pathloss_vs_distance.png"))
plt.close()

# ----------------------------------------------------------
# 4. Path Loss by Season
# ----------------------------------------------------------

plt.figure(figsize=(8,5))
sns.boxplot(
    x=df["Seasonal Variation (Data Source)"],
    y=df[target]
)
plt.xticks(rotation=45)
plt.title("Path Loss Across Seasons")
plt.savefig(os.path.join(results_dir, "pathloss_by_season.png"))
plt.close()

# ----------------------------------------------------------
# 5. Actual vs Predicted
# ----------------------------------------------------------

plt.figure(figsize=(7,7))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)
plt.xlabel("Actual Path Loss")
plt.ylabel("Predicted Path Loss")
plt.title("Actual vs Predicted")
plt.savefig(os.path.join(results_dir, "actual_vs_predicted.png"))
plt.close()

# ----------------------------------------------------------
# 6. Residual Plot
# ----------------------------------------------------------

residuals = y_test - y_pred

plt.figure(figsize=(8,5))
sns.histplot(residuals, bins=30, kde=True)
plt.title("Residual Error Distribution")
plt.xlabel("Residual Error")
plt.savefig(os.path.join(results_dir, "residual_plot.png"))
plt.close()

print("\nAll Visualizations Completed Successfully and saved to the 'results' folder!")

Saving 5g-South Asia.csv to 5g-South Asia (3).csv
Dataset Loaded Successfully!

  Seasonal Variation (Data Source)  Simulation Run Number  \
0                            FallL                      1   
1                            FallL                      1   
2                            FallL                      1   
3                            FallL                      1   
4                            FallL                      1   

   T-R Separation Distance (m)  Time Delay (ns)  Received Power (dBm)  \
0                        216.4              721                -120.8   
1                        216.4              725                -127.8   
2                        216.4              729                -119.8   
3                        216.4              735                -118.1   
4                        216.4              741                -126.1   

    Phase (rad)  Azimuth AoD (degree)  Elevation AoD (degree)  \
0           3.2                   291            

In [ ]:
import pandas as pd
import joblib

# =====================================================
# Load Saved Model
# =====================================================

model = joblib.load("5G_PathLoss_Model.pkl")

print("="*60)
print("      5G PATH LOSS PREDICTION SYSTEM")
print("="*60)

# =====================================================
# Take User Inputs
# (Ensure all inputs match the features used for training)
# =====================================================

print("Please enter the following details for prediction:")

simulation_run_number = int(input("Simulation Run Number (int): "))
distance = float(input("T-R Separation Distance (m) (float): "))
time_delay = int(input("Time Delay (ns) (int): "))
received_power = float(input("Received Power (dBm) (float): "))
phase = float(input("Phase (rad) (float): "))
azimuth_aod = int(input("Azimuth AoD (degree) (int): "))
elevation_aod = float(input("Elevation AoD (degree) (float): "))
azimuth_aoa = float(input("Azimuth AoA (degree) (float): "))
elevation_aoa = int(input("Elevation AoA (degree) (int): "))
rms_delay_spread = float(input("RMS Delay Spread (ns) (float): "))
season_numerical = int(input("Season (numerical, e.g., 0 for FallL, 1 for SpringH, etc.) (int): ")) # From original dataset, Season is numerical
frequency = float(input("Frequency (float): ")) # Corrected column name

seasonal_variation_data_source = input("Seasonal Variation (Data Source) (e.g., FallL, SummerM, WinterH): ") # Categorical input


# =====================================================
# Create DataFrame
# =====================================================

sample = pd.DataFrame({
    "Seasonal Variation (Data Source)": [seasonal_variation_data_source],
    "Simulation Run Number": [simulation_run_number],
    "T-R Separation Distance (m)": [distance],
    "Time Delay (ns)": [time_delay],
    "Received Power (dBm)": [received_power],
    "Phase (rad)": [phase],
    "Azimuth AoD (degree)": [azimuth_aod],
    "Elevation AoD (degree)": [elevation_aod],
    "Azimuth AoA (degree)": [azimuth_aoa],
    "Elevation AoA (degree)": [elevation_aoa],
    "RMS Delay Spread (ns)": [rms_delay_spread],
    "Season": [season_numerical],
    "Frequency": [frequency]
})

# =====================================================
# Prediction
# =====================================================

predicted_path_loss = model.predict(sample)[0]

# =====================================================
# Classification
# =====================================================

if predicted_path_loss < 80:
    quality = "Excellent Signal"

elif predicted_path_loss < 100:
    quality = "Good Signal"

elif predicted_path_loss < 120:
    quality = "Fair Signal"

else:
    quality = "Poor Signal"

# =====================================================
# Output
# =====================================================

print("\n" + "="*60)
print("Prediction Result")
print("="*60)

print(f"Predicted Path Loss : {predicted_path_loss:.2f} dB")
print(f"Signal Quality      : {quality}")

print("="*60)

      5G PATH LOSS PREDICTION SYSTEM
Please enter the following details for prediction:


### Run Prediction via Command Line Interface

To run the prediction from a command-line interface, we'll create a Python script `predict_cli.py` that accepts all the input features as command-line arguments. This script will then load the trained model, make a prediction, and print the result. The `argparse` module is used for robust command-line argument handling.

In [20]:
%%writefile predict_cli.py

import pandas as pd
import joblib
import argparse
import sys

# =====================================================
# Load Saved Model
# =====================================================

try:
    model = joblib.load("5G_PathLoss_Model.pkl")
except FileNotFoundError:
    print("Error: Model file '5G_PathLoss_Model.pkl' not found. Please ensure it's in the current directory.")
    sys.exit(1)

# =====================================================
# Command Line Argument Parsing
# =====================================================

def parse_args():
    parser = argparse.ArgumentParser(description="5G Path Loss Prediction System")
    parser.add_argument("--sim_run_num", type=int, required=True, help="Simulation Run Number (int)")
    parser.add_argument("--distance", type=float, required=True, help="T-R Separation Distance (m) (float)")
    parser.add_argument("--time_delay", type=int, required=True, help="Time Delay (ns) (int)")
    parser.add_argument("--received_power", type=float, required=True, help="Received Power (dBm) (float)")
    parser.add_argument("--phase", type=float, required=True, help="Phase (rad) (float)")
    parser.add_argument("--azimuth_aod", type=int, required=True, help="Azimuth AoD (degree) (int)")
    parser.add_argument("--elevation_aod", type=float, required=True, help="Elevation AoD (degree) (float)")
    parser.add_argument("--azimuth_aoa", type=float, required=True, help="Azimuth AoA (degree) (float)")
    parser.add_argument("--elevation_aoa", type=int, required=True, help="Elevation AoA (degree) (int)") # Reverted to int based on original df.dtypes
    parser.add_argument("--rms_delay_spread", type=float, required=True, help="RMS Delay Spread (ns) (float)")
    parser.add_argument("--season_num", type=int, required=True, help="Season (numerical, e.g., 0 for FallL, 1 for SpringH) (int)")
    parser.add_argument("--frequency", type=float, required=True, help="Frequency (float)")
    parser.add_argument("--seasonal_variation", type=str, required=True, help="Seasonal Variation (Data Source) (e.g., FallL, SummerM, WinterH)")
    return parser.parse_args()


def main():
    args = parse_args()

    # =====================================================
    # Create DataFrame from Arguments
    # =====================================================

    sample = pd.DataFrame({
        "Seasonal Variation (Data Source)": [args.seasonal_variation],
        "Simulation Run Number": [args.sim_run_num],
        "T-R Separation Distance (m)": [args.distance],
        "Time Delay (ns)": [args.time_delay],
        "Received Power (dBm)": [args.received_power],
        "Phase (rad)": [args.phase],
        "Azimuth AoD (degree)": [args.azimuth_aod],
        "Elevation AoD (degree)": [args.elevation_aod],
        "Azimuth AoA (degree)": [args.azimuth_aoa],
        "Elevation AoA (degree)": [args.elevation_aoa],
        "RMS Delay Spread (ns)": [args.rms_delay_spread],
        "Season": [args.season_num],
        "Frequency": [args.frequency]
    })

    # =====================================================
    # Prediction
    # =====================================================

    predicted_path_loss = model.predict(sample)[0]

    # =====================================================
    # Classification (Adjusted Thresholds)
    # =====================================================

    if predicted_path_loss < 110: # Adjusted from 80
        quality = "Excellent Signal"
    elif predicted_path_loss < 125: # Adjusted from 100
        quality = "Good Signal"
    elif predicted_path_loss < 140: # Adjusted from 120
        quality = "Fair Signal"
    else:
        quality = "Poor Signal"

    # =====================================================
    # Output
    # =====================================================

    print("\n" + "="*60)
    print("Prediction Result")
    print("="*60)
    print(f"Raw Predicted Path Loss : {predicted_path_loss:.2f} dB") # Added for debugging
    print(f"Predicted Path Loss : {predicted_path_loss:.2f} dB")
    print(f"Signal Quality      : {quality}")
    print("="*60)


if __name__ == "__main__":
    main()

Overwriting predict_cli.py


Now, you can run the `predict_cli.py` script from the command line using `!python` and passing the arguments. Replace the example values with your desired input.

In [16]:
# Example CLI execution. Replace values with your desired inputs.
!python predict_cli.py \
    --sim_run_num 12 \
    --distance 1.2 \
    --time_delay 1\
    --received_power -30 \
    --phase 0.8 \
    --azimuth_aod 10 \
    --elevation_aod 5 \
    --azimuth_aoa 20 \
    --elevation_aoa 8 \
    --rms_delay_spread 1.5 \
    --season_num 1 \
    --frequency 3.5 \
    --seasonal_variation "SummerM"


Prediction Result
Raw Predicted Path Loss : 107.94 dB
Predicted Path Loss : 107.94 dB
Signal Quality      : Fair Signal


### Test Case 1: Inputs likely to result in 'Excellent Signal' (e.g., very short distance, high received power)

In [19]:
# Example CLI execution for 'Excellent Signal'
!python predict_cli.py \
--sim_run_num 1 \
--distance 5.0 \
--time_delay 1 \
--received_power -50 \
--phase 0.1 \
--azimuth_aod 5 \
--elevation_aod 2.0 \
--azimuth_aoa 10.0 \
--elevation_aoa 3 \
--rms_delay_spread 0.5 \
--season_num 0 \
--frequency 28.0 \
--seasonal_variation "FallL"


Prediction Result
Raw Predicted Path Loss : 111.75 dB
Predicted Path Loss : 111.75 dB
Signal Quality      : Fair Signal


### Test Case 2: Inputs likely to result in 'Poor Signal' (e.g., very long distance, low received power)

In [18]:
# Example CLI execution for 'Poor Signal'
!python predict_cli.py \
    --sim_run_num 100 \
    --distance 500.0 \
    --time_delay 50 \
    --received_power -120 \
    --phase 3.0 \
    --azimuth_aod 180 \
    --elevation_aod -45.0 \
    --azimuth_aoa 270.0 \
    --elevation_aoa -60 \
    --rms_delay_spread 10.0 \
    --season_num 3 \
    --frequency 3.0 \
    --seasonal_variation "WinterH"


Prediction Result
Raw Predicted Path Loss : 149.48 dB
Predicted Path Loss : 149.48 dB
Signal Quality      : Poor Signal


In [ ]:
# Example CLI execution for 'Poor Signal'
!python predict_cli.py \
    --sim_run_num 100 \
    --distance 500.0 \
    --time_delay 50 \
    --received_power -120\
    --phase 3.0 \
    --azimuth_aod 180 \
    --elevation_aod -45.0 \
    --azimuth_aoa 270.0 \
    --elevation_aoa -60 \
    --rms_delay_spread 10.0 \
    --season_num 3 \
    --frequency 3.0 \
    --seasonal_variation "WinterH"


Prediction Result
Raw Predicted Path Loss : 149.48 dB
Predicted Path Loss : 149.48 dB
Signal Quality      : Poor Signal
